# 检索侧补充对比

- 对应论文章节：第3.3.3节 检索侧补充对比实验
- 源脚本：`experiments/03_证伪实验/scripts/运行_检索侧补充对比.py`
- notebook 作用：直接查看代码与已保存结果，命令行运行仍以 `.py` 为准

这本 notebook 对应上下文化检索、查询改写、父子块检索与基线之间的比较。

## 命令行复现

```bash
cd /root/Velo
/root/Velo/.venv/bin/python experiments/03_证伪实验/scripts/运行_检索侧补充对比.py --variant baseline
```

## 源码镜像

下面这一格保留 `.py` 的完整源码，主要用于现场查阅。

In [ ]:
"""比较检索侧替代路线在同一批复杂问答上的表现。"""

from __future__ import annotations

import argparse
import json
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
EXPERIMENTS_ROOT = ROOT.parent
IMPL_ROOT = EXPERIMENTS_ROOT / "04_算法实现"
if str(IMPL_ROOT) not in sys.path:
    sys.path.insert(0, str(IMPL_ROOT))

from retrieval_pipeline.common import DEFAULT_EMBEDDING_MODEL, DEFAULT_LLM_MODEL, ensure_dir
from retrieval_pipeline.datasets import load_crud_cases
from retrieval_pipeline.metrics import evaluate_crud_results
from retrieval_pipeline.pipeline import PipelineVariant, RagExperimentPipeline

OUTPUT_ROOT = ROOT / "results" / "03_检索侧补充对比"

COMPLEX_CASE_IDS = (
    "questanswer_2docs_002",
    "questanswer_2docs_003",
    "questanswer_2docs_005",
    "questanswer_2docs_006",
    "questanswer_2docs_008",
    "questanswer_3docs_003",
    "questanswer_3docs_004",
    "questanswer_3docs_006",
)

VARIANTS = {
    "baseline": PipelineVariant(
        key="baseline_rrf_rerank_direct",
        label="基线",
        use_rerank=True,
        answer_prompt_style="simple",
        multi_snippet_count=2,
    ),
    "contextual": PipelineVariant(
        key="contextual_rrf_rerank_direct",
        label="上下文化检索",
        retrieval_strategy="contextual",
        use_rerank=True,
        answer_prompt_style="simple",
        multi_snippet_count=2,
    ),
    "rewrite": PipelineVariant(
        key="rewrite_rrf_rerank_direct",
        label="查询改写",
        use_rerank=True,
        use_query_rewrite=True,
        answer_prompt_style="simple",
        multi_snippet_count=2,
    ),
    "parent_child": PipelineVariant(
        key="parent_child_rrf_rerank_direct",
        label="父子块检索",
        retrieval_strategy="parent_child",
        use_rerank=True,
        answer_prompt_style="simple",
        multi_snippet_count=2,
    ),
}

SUMMARY_FILE_NAMES = {
    "baseline": "检索侧_基线_结果汇总.json",
    "contextual": "检索侧_上下文化检索_结果汇总.json",
    "rewrite": "检索侧_查询改写_结果汇总.json",
    "parent_child": "检索侧_父子块检索_结果汇总.json",
}

DETAIL_FILE_NAMES = {
    "baseline": "检索侧_基线_逐题明细.json",
    "contextual": "检索侧_上下文化检索_逐题明细.json",
    "rewrite": "检索侧_查询改写_逐题明细.json",
    "parent_child": "检索侧_父子块检索_逐题明细.json",
}


def run_variant_on_cases(pipeline: RagExperimentPipeline, prepared, variant: PipelineVariant):
    results = []
    total = len(prepared.cases)
    for index, case in enumerate(prepared.cases, start=1):
        if index == 1 or index == total:
            print(f"[retrieval-compare] {variant.key}: {index}/{total}", flush=True)
        results.append(pipeline.run_case(prepared, case, variant))
    return results


def load_complex_cases():
    cases, docs = load_crud_cases(
        summary_samples=6,
        qa_1doc_samples=6,
        qa_2doc_samples=8,
        qa_3doc_samples=8,
        hallu_samples=4,
        negative_samples=6,
        distractor_count=600,
        seed=42,
    )
    selected_cases = [case for case in cases if case.case_id in COMPLEX_CASE_IDS]
    if len(selected_cases) != len(COMPLEX_CASE_IDS):
        found = {case.case_id for case in selected_cases}
        missing = [case_id for case_id in COMPLEX_CASE_IDS if case_id not in found]
        raise RuntimeError(f"缺少复杂验证样例: {missing}")
    return selected_cases, docs


def main() -> None:
    parser = argparse.ArgumentParser(description="运行检索侧补充对比实验。")
    parser.add_argument("--variant", choices=tuple(VARIANTS), required=True)
    parser.add_argument("--llm-model", default=DEFAULT_LLM_MODEL)
    args = parser.parse_args()

    ensure_dir(OUTPUT_ROOT)
    cache_root = ensure_dir(ROOT / ".cache")

    selected_cases, docs = load_complex_cases()
    variant = VARIANTS[args.variant]
    pipeline = RagExperimentPipeline(
        cache_root=cache_root,
        embedding_model=DEFAULT_EMBEDDING_MODEL,
        llm_model=args.llm_model,
    )
    prepared = pipeline.prepare_dataset(
        "crud_retrieval_compare_batch",
        selected_cases,
        docs,
        include_contextual=variant.retrieval_strategy == "contextual",
        include_parent_child=variant.retrieval_strategy == "parent_child",
        include_query_rewrite=variant.use_query_rewrite,
    )
    results = run_variant_on_cases(pipeline, prepared, variant)
    evaluation = evaluate_crud_results(
        variant.key,
        results,
        selected_cases,
        ragas_case_ids=(),
        qa_ragas_case_ids=(),
        multidoc_ragas_case_ids=(),
        enable_ragas=False,
    )
    summary = dict(evaluation.summary)
    summary["label"] = variant.label
    payload = {"case_ids": list(COMPLEX_CASE_IDS), "summaries": [summary]}
    suffix = args.variant
    (OUTPUT_ROOT / SUMMARY_FILE_NAMES[suffix]).write_text(
        json.dumps(payload, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    (OUTPUT_ROOT / DETAIL_FILE_NAMES[suffix]).write_text(
        json.dumps(evaluation.detail_rows, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    print(json.dumps(payload, ensure_ascii=False, indent=2))


if __name__ == "__main__":
    main()


## 结果预览

下面直接内嵌当前已保存结果的关键文件预览。

### 检索侧：基线

- 文件：`../results/03_检索侧补充对比/检索侧_基线_结果汇总.json`

In [1]:
from pathlib import Path
import json

path = Path('../results/03_检索侧补充对比/检索侧_基线_结果汇总.json')
data = json.loads(path.read_text(encoding='utf-8'))
if isinstance(data, list) and len(data) > 6:
    data = {'total_items': len(data), 'preview': data[:6]}
elif isinstance(data, dict):
    data = dict(data)
    for key in ('summaries', 'preview', 'rows'):
        value = data.get(key)
        if isinstance(value, list) and len(value) > 6:
            data[key] = {'total_items': len(value), 'preview': value[:6]}
print(json.dumps(data, ensure_ascii=False, indent=2))


{
  "case_ids": [
    "questanswer_2docs_002",
    "questanswer_2docs_003",
    "questanswer_2docs_005",
    "questanswer_2docs_006",
    "questanswer_2docs_008",
    "questanswer_3docs_003",
    "questanswer_3docs_004",
    "questanswer_3docs_006"
  ],
  "summaries": [
    {
      "variant": "baseline_rrf_rerank_direct",
      "dataset": "crud",
      "faithfulness": 0.0,
      "answer_correctness": 0.0,
      "answer_relevancy": 0.0,
      "context_precision": 0.0,
      "ragas_sample_count": 0,
      "accuracy": 0.0,
      "qa_accuracy": 0.0,
      "retrieval_hit_rate_at_1": 0.875,
      "retrieval_hit_rate_at_3": 1.0,
      "qa_faithfulness": 0.0,
      "qa_answer_correctness": 0.0,
      "qa_answer_relevancy": 0.0,
      "qa_context_precision": 0.0,
      "qa_ragas_sample_count": 0,
      "multidoc_faithfulness": 0.0,
      "multidoc_answer_correctness": 0.0,
      "multidoc_answer_relevancy": 0.0,
      "multidoc_context_precision": 0.0,
      "multidoc_ragas_sample_count": 0,
  

### 检索侧：上下文化检索

- 文件：`../results/03_检索侧补充对比/检索侧_上下文化检索_结果汇总.json`

In [2]:
from pathlib import Path
import json

path = Path('../results/03_检索侧补充对比/检索侧_上下文化检索_结果汇总.json')
data = json.loads(path.read_text(encoding='utf-8'))
if isinstance(data, list) and len(data) > 6:
    data = {'total_items': len(data), 'preview': data[:6]}
elif isinstance(data, dict):
    data = dict(data)
    for key in ('summaries', 'preview', 'rows'):
        value = data.get(key)
        if isinstance(value, list) and len(value) > 6:
            data[key] = {'total_items': len(value), 'preview': value[:6]}
print(json.dumps(data, ensure_ascii=False, indent=2))


{
  "case_ids": [
    "questanswer_2docs_002",
    "questanswer_2docs_003",
    "questanswer_2docs_005",
    "questanswer_2docs_006",
    "questanswer_2docs_008",
    "questanswer_3docs_003",
    "questanswer_3docs_004",
    "questanswer_3docs_006"
  ],
  "summaries": [
    {
      "variant": "contextual_rrf_rerank_direct",
      "dataset": "crud",
      "faithfulness": 0.0,
      "answer_correctness": 0.0,
      "answer_relevancy": 0.0,
      "context_precision": 0.0,
      "ragas_sample_count": 0,
      "accuracy": 0.0,
      "qa_accuracy": 0.0,
      "retrieval_hit_rate_at_1": 0.875,
      "retrieval_hit_rate_at_3": 1.0,
      "qa_faithfulness": 0.0,
      "qa_answer_correctness": 0.0,
      "qa_answer_relevancy": 0.0,
      "qa_context_precision": 0.0,
      "qa_ragas_sample_count": 0,
      "multidoc_faithfulness": 0.0,
      "multidoc_answer_correctness": 0.0,
      "multidoc_answer_relevancy": 0.0,
      "multidoc_context_precision": 0.0,
      "multidoc_ragas_sample_count": 0,


### 检索侧：查询改写

- 文件：`../results/03_检索侧补充对比/检索侧_查询改写_结果汇总.json`

In [3]:
from pathlib import Path
import json

path = Path('../results/03_检索侧补充对比/检索侧_查询改写_结果汇总.json')
data = json.loads(path.read_text(encoding='utf-8'))
if isinstance(data, list) and len(data) > 6:
    data = {'total_items': len(data), 'preview': data[:6]}
elif isinstance(data, dict):
    data = dict(data)
    for key in ('summaries', 'preview', 'rows'):
        value = data.get(key)
        if isinstance(value, list) and len(value) > 6:
            data[key] = {'total_items': len(value), 'preview': value[:6]}
print(json.dumps(data, ensure_ascii=False, indent=2))


{
  "case_ids": [
    "questanswer_2docs_002",
    "questanswer_2docs_003",
    "questanswer_2docs_005",
    "questanswer_2docs_006",
    "questanswer_2docs_008",
    "questanswer_3docs_003",
    "questanswer_3docs_004",
    "questanswer_3docs_006"
  ],
  "summaries": [
    {
      "variant": "rewrite_rrf_rerank_direct",
      "dataset": "crud",
      "faithfulness": 0.0,
      "answer_correctness": 0.0,
      "answer_relevancy": 0.0,
      "context_precision": 0.0,
      "ragas_sample_count": 0,
      "accuracy": 0.0,
      "qa_accuracy": 0.0,
      "retrieval_hit_rate_at_1": 0.875,
      "retrieval_hit_rate_at_3": 1.0,
      "qa_faithfulness": 0.0,
      "qa_answer_correctness": 0.0,
      "qa_answer_relevancy": 0.0,
      "qa_context_precision": 0.0,
      "qa_ragas_sample_count": 0,
      "multidoc_faithfulness": 0.0,
      "multidoc_answer_correctness": 0.0,
      "multidoc_answer_relevancy": 0.0,
      "multidoc_context_precision": 0.0,
      "multidoc_ragas_sample_count": 0,
   

### 检索侧：父子块检索

- 文件：`../results/03_检索侧补充对比/检索侧_父子块检索_结果汇总.json`

In [4]:
from pathlib import Path
import json

path = Path('../results/03_检索侧补充对比/检索侧_父子块检索_结果汇总.json')
data = json.loads(path.read_text(encoding='utf-8'))
if isinstance(data, list) and len(data) > 6:
    data = {'total_items': len(data), 'preview': data[:6]}
elif isinstance(data, dict):
    data = dict(data)
    for key in ('summaries', 'preview', 'rows'):
        value = data.get(key)
        if isinstance(value, list) and len(value) > 6:
            data[key] = {'total_items': len(value), 'preview': value[:6]}
print(json.dumps(data, ensure_ascii=False, indent=2))


{
  "case_ids": [
    "questanswer_2docs_002",
    "questanswer_2docs_003",
    "questanswer_2docs_005",
    "questanswer_2docs_006",
    "questanswer_2docs_008",
    "questanswer_3docs_003",
    "questanswer_3docs_004",
    "questanswer_3docs_006"
  ],
  "summaries": [
    {
      "variant": "parent_child_rrf_rerank_direct",
      "dataset": "crud",
      "faithfulness": 0.0,
      "answer_correctness": 0.0,
      "answer_relevancy": 0.0,
      "context_precision": 0.0,
      "ragas_sample_count": 0,
      "accuracy": 0.0,
      "qa_accuracy": 0.0,
      "retrieval_hit_rate_at_1": 0.75,
      "retrieval_hit_rate_at_3": 0.875,
      "qa_faithfulness": 0.0,
      "qa_answer_correctness": 0.0,
      "qa_answer_relevancy": 0.0,
      "qa_context_precision": 0.0,
      "qa_ragas_sample_count": 0,
      "multidoc_faithfulness": 0.0,
      "multidoc_answer_correctness": 0.0,
      "multidoc_answer_relevancy": 0.0,
      "multidoc_context_precision": 0.0,
      "multidoc_ragas_sample_count": 